In [17]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [18]:
pip install dagshub mlflow scikit-learn pandas matplotlib seaborn skops --quiet

Note: you may need to restart the kernel to use updated packages.


In [19]:
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

os.environ['MLFLOW_TRACKING_USERNAME'] = 'lchkh23'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '24b59ad62b5f1811d850606de6be1fe2fb3dfe05'

dagshub.init(repo_owner='lchkh23', repo_name='Housing-Prices', mlflow=True)

Initialized MLflow to track repo "lchkh23/Housing-Prices"

Repository lchkh23/Housing-Prices initialized!

# LOAD MODEL 

In [20]:
RUN_ID = 'b04579db90b841e58346739060beb73d'

model  = mlflow.sklearn.load_model("models:/house-prices-linear/latest")
scaler = mlflow.sklearn.load_model("models:/house-prices-scaler/latest")

print('loaded successfully')

loaded successfully


# LOAD DATA

In [21]:
train = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

test_ids = test['Id']

x = train.drop(columns=['SalePrice'])
y = np.log1p(train['SalePrice'])

print(f'Train: {x.shape}, Test: {test.shape}')

Train: (1460, 80), Test: (1459, 80)


# DROPPING HIGH NAN COLS

In [22]:
missing_vals = x.isnull().mean() * 100
to_drop_cols = missing_vals[missing_vals > 80].index

x    = x.drop(columns=to_drop_cols)
test = test.drop(columns=to_drop_cols, errors='ignore')
print(f'Dropped high NaN cols: {to_drop_cols.tolist()}')

Dropped high NaN cols: ['Alley', 'PoolQC', 'Fence', 'MiscFeature']


# IMPUTATION

In [23]:
numeric_cols     = x.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = x.select_dtypes(include=['object']).columns

train_medians = x[numeric_cols].median()
train_modes   = {col: x[col].mode()[0] for col in categorical_cols}

x[numeric_cols] = x[numeric_cols].fillna(train_medians)
for col in categorical_cols:
    x[col] = x[col].fillna(train_modes[col])

for col in test.select_dtypes(include=['int64', 'float64']).columns:
    if col in train_medians:
        test[col] = test[col].fillna(train_medians[col])
for col in test.select_dtypes(include=['object']).columns:
    if col in train_modes:
        test[col] = test[col].fillna(train_modes[col])

# DROPPING USELESS COLUMSN

In [24]:
low_variance_cols = [
    'Utilities', 'Street', 'PoolArea', 'Condition2', 'RoofMatl',
    '3SsnPorch', 'LowQualFinSF', 'Heating', 'GarageCond', 'MiscVal',
    'GarageQual', 'LandSlope', 'BsmtHalfBath', 'CentralAir', 'Functional',
    'BsmtCond', 'Electrical', 'ScreenPorch', 'PavedDrive', 'LandContour'
]

x    = x.drop(columns=low_variance_cols, errors='ignore')
test = test.drop(columns=low_variance_cols, errors='ignore')
print(f'After variance drop - Train: {x.shape}, Test: {test.shape}')

After variance drop - Train: (1460, 56), Test: (1459, 56)


# FEATURE ENGINEERING

In [25]:
for df in [x, test]:
    df['TotalSF']      = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath']    = df['FullBath'] + 0.5*df['HalfBath'] + df['BsmtFullBath']
    df['HouseAge']     = df['YrSold'] - df['YearBuilt']
    df['Remodeled']    = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)
    df['TotalPorchSF'] = df['OpenPorchSF'] + df['WoodDeckSF'] + df['EnclosedPorch']

cols_to_drop = [
    'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
    'FullBath', 'HalfBath', 'BsmtFullBath',
    'YearBuilt', 'YearRemodAdd', 'YrSold',
    'OpenPorchSF', 'WoodDeckSF', 'EnclosedPorch',
    'Id'
]

x    = x.drop(columns=cols_to_drop, errors='ignore')
test = test.drop(columns=cols_to_drop, errors='ignore')
print(f'After feature engineering - Train: {x.shape}, Test: {test.shape}')

After feature engineering - Train: (1460, 48), Test: (1459, 48)


# ORDINAL ENCODING

In [26]:
quality_map  = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
exposure_map = {'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
bsmt_map     = {'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
garage_map   = {'Unf': 1, 'RFn': 2, 'Fin': 3}

qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'HeatingQC', 'KitchenQual', 'FireplaceQu']

for df in [x, test]:
    for col in qual_cols:
        df[col] = df[col].map(quality_map)
    df['BsmtExposure'] = df['BsmtExposure'].map(exposure_map)
    df['BsmtFinType1'] = df['BsmtFinType1'].map(bsmt_map)
    df['BsmtFinType2'] = df['BsmtFinType2'].map(bsmt_map)
    df['GarageFinish'] = df['GarageFinish'].map(garage_map)

print('Ordinal encoding done')

Ordinal encoding done


In [28]:
temp = x[['Neighborhood']].copy()
temp['SalePrice'] = y.values
neighborhood_means = temp.groupby('Neighborhood')['SalePrice'].mean()

x['Neighborhood']    = x['Neighborhood'].map(neighborhood_means)
test['Neighborhood'] = test['Neighborhood'].map(neighborhood_means)
test['Neighborhood'] = test['Neighborhood'].fillna(y.mean())

print('Target encoding done')

Target encoding done


# ONE HOT ENCODING

In [29]:
cat_cols = x.select_dtypes(include=['object']).columns.tolist()

x    = pd.get_dummies(x,    columns=cat_cols, drop_first=True)
test = pd.get_dummies(test, columns=cat_cols, drop_first=True)

x, test = x.align(test, join='left', axis=1, fill_value=0)
print(f'After OHE - Train: {x.shape}, Test: {test.shape}')

After OHE - Train: (1460, 123), Test: (1459, 123)


# SCALING

In [30]:
x_scaled    = scaler.transform(x)
test_scaled = scaler.transform(test)
print('Scaling done')

Scaling done


# RFE

In [31]:
estimator = LinearRegression()
rfe = RFE(estimator, n_features_to_select=40)
rfe.fit(x_scaled, y)

x_rfe         = rfe.transform(x_scaled)
test_rfe      = rfe.transform(test_scaled)
selected_cols = x.columns[rfe.support_].tolist()
print(f'RFE selected {len(selected_cols)} features')

RFE selected 40 features


# CORRELATION FILTER

In [38]:
x_rfe_df    = pd.DataFrame(x_rfe,    columns=selected_cols)
test_rfe_df = pd.DataFrame(test_rfe, columns=selected_cols)

corr_matrix  = x_rfe_df.corr().abs()
upper        = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.85)]
print('Dropping correlated cols:', to_drop_corr)

x_final    = x_rfe_df.drop(columns=to_drop_corr)
test_final = test_rfe_df.drop(columns=to_drop_corr)
print(f'Final shape - Train: {x_final.shape}, Test: {test_final.shape}')

Dropping correlated cols: ['GarageArea', 'Exterior2nd_Wd Sdng', 'SaleCondition_Partial']
Final shape - Train: (1460, 37), Test: (1459, 37)


# PREDICTION

In [40]:
test_final = test_final.reindex(columns=model.feature_names_in_, fill_value=0)
test_preds_log = model.predict(test_final)
test_preds     = np.expm1(test_preds_log)

print(f'Price range: ${test_preds.min():,.0f} - ${test_preds.max():,.0f}')

Price range: $50,292 - $714,030


In [41]:
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_preds
})

submission.to_csv('submission.csv', index=False)
print('submission.csv saved!')
print(submission.head(10))

submission.csv saved!
     Id      SalePrice
0  1461  105057.718180
1  1462  140276.382133
2  1463  171063.043795
3  1464  194455.748977
4  1465  205600.782416
5  1466  174369.586607
6  1467  185748.894403
7  1468  168574.389493
8  1469  200375.538604
9  1470  111452.950063
